# Motor-Imagery EEG — LOSO benchmark on a Kaggle GPU

Subject-independent (**Leave-One-Subject-Out**) benchmark — the honest number. LOSO is the most interesting test for Denoising-EEGNet: cross-subject noise is exactly what its front-end targets.

### Before you run — two toggles (right-hand panel)
1. **Settings ▸ Accelerator ▸ GPU** (T4 or P100).
2. **Settings ▸ Internet ▸ On** — required for `git clone`, `pip install`, and the dataset download.

### Why LOSO is slow, and the knobs that fix it
LOSO retrains every model on ~all other subjects, **once per held-out subject** (40 subjects → 40 folds). The classical models are near-instant; the two deep nets at 100 epochs each are the bottleneck (~80 deep fits). The config cell below therefore **cuts deep epochs 100→40 and raises batch 32→64**, and you can shrink `N_SUBJECTS`. For an answer in *minutes*, run the **classical-only** cell first.

Use **Save Version ▸ Save & Run All (Commit)** to run headless — no need to keep the tab open.

## 1. Confirm GPU + internet

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — enable Settings > Accelerator > GPU'
import urllib.request
try:
    urllib.request.urlopen('https://physionet.org', timeout=10); print('internet: OK')
except Exception as e:
    print('internet: OFF — enable Settings > Internet >', e)

## 2. Clone the repo (into /kaggle/working)

In [ ]:
import os
os.chdir('/kaggle/working')
REPO = 'https://github.com/ol1sa/Motor-Imagery-EEG-.git'
if not os.path.isdir('Motor-Imagery-EEG-'):
    !git clone -q $REPO
os.chdir('/kaggle/working/Motor-Imagery-EEG-')
!git pull -q
print('cwd:', os.getcwd())

## 3. Install only what Kaggle lacks
Kaggle ships GPU PyTorch + the scientific stack. We add MNE, pyRiemann and statsmodels and **don't** touch torch/NumPy.

In [ ]:
!pip install -q mne pyriemann statsmodels pooch
import torch, mne, pyriemann
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('mne', mne.__version__, '| pyriemann', pyriemann.__version__)

## 4. Speed knobs + config builder
Lower `N_SUBJECTS` for a quicker pass (it cuts both the number of folds *and* the size of each training set, so the saving is roughly quadratic).

In [ ]:
import yaml

N_SUBJECTS  = 40      # 40 = comparable to the within-subject table; 20 ~= 4x faster
DEEP_EPOCHS = 40      # was 100; LOSO trains on lots of data so fewer epochs suffice
BATCH       = 64      # was 32; larger batch uses the GPU better

def make_config(name, models):
    cfg = yaml.safe_load(open('configs/binary.yaml'))
    cfg['name'] = name
    cfg['data']['subjects'] = list(range(1, N_SUBJECTS + 1))
    cfg['model']['names'] = models
    cfg['model']['deep']['epochs'] = DEEP_EPOCHS
    cfg['model']['deep']['batch_size'] = BATCH
    path = f'configs/{name}.yaml'
    yaml.safe_dump(cfg, open(path, 'w'), sort_keys=False)
    return path

print('knobs:', N_SUBJECTS, 'subjects | deep epochs', DEEP_EPOCHS, '| batch', BATCH)

## 5a. FAST: classical-only LOSO (3 models, minutes)
Run this first — CSP/Riemann under LOSO are seconds per fold, so you get 3 of the 5 LOSO numbers almost immediately (after the one-time download/preprocess). Tag: `kclf_binary_loso`.

In [ ]:
cfg_clf = make_config('kclf', ['csp_lda', 'csp_svm', 'riemann_lr'])
!PYTHONPATH=src python -u -m mibci.run --config $cfg_clf --experiment binary --cv loso

## 5b. FULL: all 5 models LOSO (adds the two deep nets)
Reuses the cached, already-preprocessed epochs from 5a, so this only pays for model training. Tag: `kaggle_binary_loso`.

In [ ]:
cfg_all = make_config('kaggle', ['csp_lda', 'csp_svm', 'riemann_lr', 'eegnet', 'denoising_eegnet'])
!PYTHONPATH=src python -u -m mibci.run --config $cfg_all --experiment binary --cv loso

## 6. Show the LOSO results

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
for tag in ['kclf_binary_loso', 'kaggle_binary_loso']:
    if glob.glob(f'artifacts/summary_{tag}.csv'):
        print(f'\n===== {tag} =====')
        display(pd.read_csv(f'artifacts/summary_{tag}.csv'))
        display(pd.read_csv(f'artifacts/stats_{tag}.csv'))
for png in sorted(glob.glob('artifacts/*kaggle_binary_loso*.png')):
    print(png); display(Image(png))

## 7. Save results for download
Everything copied into `/kaggle/working` is saved as the notebook's **Output** and downloadable. We also zip it.

In [ ]:
import shutil, glob, os
os.makedirs('/kaggle/working/results', exist_ok=True)
for f in glob.glob('artifacts/*kclf_binary_loso*') + glob.glob('artifacts/*kaggle_binary_loso*'):
    shutil.copy(f, '/kaggle/working/results/')
shutil.make_archive('/kaggle/working/mibci_kaggle_loso_results', 'zip', '/kaggle/working/results')
print('saved:', sorted(os.listdir('/kaggle/working/results')))

In [ ]:
# (optional) push results straight back to GitHub instead of downloading.
# Add a PAT as a Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets).
#
# from kaggle_secrets import UserSecretsClient
# TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
# !git config user.name 'ol1sa' && git config user.email 'olisaogbue@icloud.com'
# !git add -f artifacts/*kclf_binary_loso* artifacts/*kaggle_binary_loso*
# !git commit -q -m 'Add Kaggle LOSO benchmark results'
# !git push -q https://$TOKEN@github.com/ol1sa/Motor-Imagery-EEG-.git main
# print('pushed')